# Delta-Gamma Variance Enhanced Options Model (Zhao & Palomar 2018)

This notebook implements the **delta-gamma variance enhanced** strategy from:

> *"A Markowitz Portfolio Approach to Options Trading"* — Licheng Zhao & Daniel P. Palomar, IEEE Trans. Signal Processing, 2018.

We model option expected returns and variance using first- and second-order Taylor expansion (delta-gamma approximation) under **lognormal GBM** assumptions. Greeks are computed from Black-Scholes using SPY chain data (IV from market or BS inversion).

**Goals:**
1. Implement the model exactly as in the paper with explicit assumptions
2. Unit tests for the model
3. §6-style robustness test: predicted vs realized (compare to `distribution.ipynb` GMM model)

---
## 1. Setup and imports

In [11]:
%matplotlib inline
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import SPY_DAILY_FILE, OPTION_CHAINS_DIR
from data.forecasts import (
    _bs_call_price, _bs_put_price,
    _bs_delta, _bs_gamma, _bs_theta,
    _bs_implied_vol,
    load_chain_for_expiry,
)

print("Imports OK")

Imports OK


---
## 2. Paper assumptions (explicitly stated)

**Assumption 1 (GBM):** The stock price $S_t$ satisfies geometric Brownian motion:
$$dS_t = \mu S_t \, dt + \sigma S_t \, dz_t$$
where $\mu$ = drift (expected return), $\sigma$ = volatility, $z_t$ = Wiener process.

**Assumption 2:** Derivative price $F_t = F_t(S_t, t)$ depends only on underlying and time.

**Assumption 3 (discrete time):** For short $\Delta t$:
- Underlying: $\Delta S_t \approx \mu S_t \Delta t + \sigma S_t \Delta z_t$
- Derivative (Itô): $\Delta F_t \approx \left(\frac{\partial F}{\partial S}\mu S + \frac{\partial F}{\partial t} + \frac{1}{2}\frac{\partial^2 F}{\partial S^2}\sigma^2 S^2\right)\Delta t + \frac{\partial F}{\partial S}\sigma S \Delta z_t$

**Assumption 4:** $\mu$ and $\sigma$ stay constant over short $\Delta t$; estimated from historical prices.

**Assumption 5:** Higher-order Greeks (beyond $\Delta$, $\Theta$, $\Gamma$) are ignored.

**Assumption 6:** Options priced via Black-Scholes (European); Greeks from BS formulas. American options approximated by European.

---
## 3. Delta-gamma model: expected returns and variance

In [12]:
def delta_gamma_expected_return(
    mu: float,
    sigma: float,
    S: float,
    delta: float,
    theta: float,
    gamma: float,
    price: float,
    dt: float = 1.0 / 252.0,
) -> float:
    """
    Expected option return over dt (paper eq. 8–9).
    E[ΔC/C] = (1/C) * (Δ*μ*S + Θ + ½*Γ*σ²*S²) * dt
    """
    if price <= 0:
        return 0.0
    drift = delta * mu * S + theta + 0.5 * gamma * (sigma ** 2) * (S ** 2)
    return (drift / price) * dt


def delta_gamma_exposure(delta: float, S: float, price: float) -> float:
    """
    Exposure to underlying: v_i = Δ*S / price (paper eq. 19, vi vector).
    Stock has exposure 1; option has exposure Δ*S/C.
    """
    if price <= 0:
        return 0.0
    return delta * S / price


def build_delta_gamma_moments(
    spot: float,
    mu: float,
    sigma: float,
    r: float,
    T: float,
    K_call: float,
    K_put: float,
    C0: float,
    P0: float,
    dt: float = 1.0 / 252.0,
    sigma_greeks: float | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Build μ (expected returns) and Σ (covariance) for [underlying, call, put, cash].
    Uses paper formulas with GBM and delta-gamma approximation.
    sigma: underlying process vol (for variance and drift ½Γσ²S²).
    sigma_greeks: vol for BS Greeks (default=sigma; use IV for market-implied).
    """
    sig_g = sigma_greeks if sigma_greeks is not None else sigma
    # Greeks from Black-Scholes (use sig_g, typically IV)
    delta_c = _bs_delta(spot, K_call, r, T, sig_g, is_call=True)
    delta_p = _bs_delta(spot, K_put, r, T, sig_g, is_call=False)
    gamma_c = _bs_gamma(spot, K_call, r, T, sig_g)
    gamma_p = _bs_gamma(spot, K_put, r, T, sig_g)
    theta_c = _bs_theta(spot, K_call, r, T, sig_g, is_call=True)
    theta_p = _bs_theta(spot, K_put, r, T, sig_g, is_call=False)

    # Expected returns (paper eq. 16–18)
    mu_und = mu * dt
    mu_call = delta_gamma_expected_return(mu, sigma, spot, delta_c, theta_c, gamma_c, C0, dt)
    mu_put = delta_gamma_expected_return(mu, sigma, spot, delta_p, theta_p, gamma_p, P0, dt)
    mu_cash = np.exp(r * dt) - 1.0

    mu_vec = np.array([mu_und, mu_call, mu_put, mu_cash])

    # Exposure vector v (paper eq. 19): stock=1, call=Δ_c*S/C, put=Δ_p*S/P, cash=0
    v_und = 1.0
    v_call = delta_gamma_exposure(delta_c, spot, C0)
    v_put = delta_gamma_exposure(delta_p, spot, P0)
    v_cash = 0.0

    v = np.array([v_und, v_call, v_put, v_cash])
    # Σ_underlying = σ² * dt (single stock)
    sigma2_und = (sigma ** 2) * dt
    Sigma_und = np.array([[sigma2_und]])  # 1x1
    # Covariance of returns: Var = w^T V Σ V^T w; here V is (4,1), Σ is (1,1)
    Sigma = np.outer(v, v) * sigma2_und
    Sigma[3, :] = 0.0
    Sigma[:, 3] = 0.0
    Sigma[3, 3] = 1e-12  # cash deterministic

    return mu_vec, Sigma

---
## 4. Load chain and compute Greeks

In [13]:
def load_chain_atm(chain_date: str, spot: float, underlying: str = "SPY") -> dict:
    """Load calls/puts via load_chain_for_expiry, pick ATM, compute IV."""
    calls, puts, expiry_used = load_chain_for_expiry(
        chain_date=chain_date, underlying=underlying, min_mid=0.05
    )
    if len(calls) == 0 or len(puts) == 0:
        raise ValueError(f"No options for {chain_date}")

    atm_call = calls.loc[(calls["strike"] - spot).abs().idxmin()]
    atm_put = puts.loc[(puts["strike"] - spot).abs().idxmin()]

    K_call = float(atm_call["strike"])
    K_put = float(atm_put["strike"])
    C0 = max(float(atm_call["mid"]), 0.10)
    P0 = max(float(atm_put["mid"]), 0.10)

    T = (pd.Timestamp(expiry_used) - pd.Timestamp(chain_date)).days / 365.0
    r = 0.05

    iv_c = atm_call.get("impl_vol", np.nan)
    if not np.isfinite(iv_c) or iv_c <= 0:
        iv_c = _bs_implied_vol(C0, spot, K_call, r, T, is_call=True) or 0.15
    iv_p = atm_put.get("impl_vol", np.nan)
    if not np.isfinite(iv_p) or iv_p <= 0:
        iv_p = _bs_implied_vol(P0, spot, K_put, r, T, is_call=False) or 0.15
    iv = float(np.mean([iv_c, iv_p]))

    return {
        "K_call": K_call, "K_put": K_put,
        "C0": C0, "P0": P0,
        "iv": iv, "expiry": expiry_used,
        "calls": calls, "puts": puts,
    }

---
## 5. Unit tests

In [14]:
def test_bs_greeks_consistency():
    """BS call/put price change ≈ delta * dS + 0.5*gamma*dS² + theta*dt."""
    S, K, r, T, sigma = 100.0, 100.0, 0.05, 30 / 365.0, 0.20
    dS, dt = 1.0, 1 / 252.0
    delta_c = _bs_delta(S, K, r, T, sigma, is_call=True)
    gamma_c = _bs_gamma(S, K, r, T, sigma)
    theta_c = _bs_theta(S, K, r, T, sigma, is_call=True)
    C0 = _bs_call_price(S, K, r, T, sigma)
    C1 = _bs_call_price(S + dS, K, r, T - dt, sigma)
    dC_exact = C1 - C0
    dC_approx = delta_c * dS + 0.5 * gamma_c * (dS ** 2) + theta_c * dt
    err = abs(dC_exact - dC_approx)
    assert err < 0.5, f"Delta-gamma approx error {err} too large"
    print("  ✓ BS Greeks delta-gamma approximation OK")


def test_exposure_scaling():
    """Option variance = (Δ*S/C)² * σ²_underlying."""
    spot, sigma, r, T = 100.0, 0.20, 0.05, 30 / 365.0
    K = 100.0
    C0 = _bs_call_price(spot, K, r, T, sigma)
    delta = _bs_delta(spot, K, r, T, sigma, is_call=True)
    v = delta_gamma_exposure(delta, spot, C0)
    dt = 1 / 252.0
    var_und = (sigma ** 2) * dt
    var_call = (v ** 2) * var_und
    assert var_call > var_und, "Call should have higher variance than underlying"
    print(f"  ✓ Exposure v={v:.2f}, var_call/var_und={var_call/var_und:.1f}")


def test_mu_sigma_shape():
    """build_delta_gamma_moments returns (4,) and (4,4)."""
    mu, Sigma = build_delta_gamma_moments(
        spot=100.0, mu=0.08, sigma=0.20, r=0.05, T=30 / 365.0,
        K_call=100.0, K_put=100.0, C0=3.0, P0=2.5,
    )
    assert mu.shape == (4,), f"mu shape {mu.shape}"
    assert Sigma.shape == (4, 4), f"Sigma shape {Sigma.shape}"
    assert np.allclose(Sigma, Sigma.T), "Sigma must be symmetric"
    print("  ✓ mu (4,), Sigma (4,4) symmetric")


print("Running unit tests...")
test_bs_greeks_consistency()
test_exposure_scaling()
test_mu_sigma_shape()
print("All tests passed.")

Running unit tests...
  ✓ BS Greeks delta-gamma approximation OK
  ✓ Exposure v=21.66, var_call/var_und=469.0
  ✓ mu (4,), Sigma (4,4) symmetric
All tests passed.


---
## 6. Single-period robustness test (predicted vs realized)

Same structure as §6 in `distribution.ipynb`: pick entry date ~35 days ago, compute predicted μ and Σ from delta-gamma model, compare to realized returns and daily-based realized covariance.

In [15]:
import os
from scipy.optimize import brentq

ASSET_NAMES = ["underlying", "call", "put", "cash"]
HOLDING_DAYS_TARGET = 1

# --- 1) Find chain date ~35 days ago ---
spy_df = pd.read_parquet(SPY_DAILY_FILE)
spy_df.index = pd.to_datetime(spy_df.index).tz_localize(None).normalize()

all_files = sorted(os.listdir(OPTION_CHAINS_DIR))
chain_dates = sorted({
    f.replace("calls_", "").replace("puts_", "").replace(".parquet", "")
    for f in all_files if f.startswith("calls_") or f.startswith("puts_")
})
today = pd.Timestamp.now().normalize()
target_entry = today - pd.Timedelta(days=HOLDING_DAYS_TARGET)
entry_date = min(chain_dates, key=lambda d: abs((pd.Timestamp(d) - target_entry).days))
entry_ts = pd.Timestamp(entry_date)
holding_days = (today - entry_ts).days
print(f"Entry date: {entry_date}  (today={today.date()}, holding={holding_days} days)", flush=True)

# --- 2) Spot and chain at entry ---
idx = spy_df.index.get_indexer([entry_ts], method="ffill")[0]
spot_entry = float(spy_df.iloc[idx]["close"])
chain = load_chain_atm(str(entry_date), spot_entry)
K_call = chain["K_call"]
K_put = chain["K_put"]
C0_mkt = chain["C0"]
P0_mkt = chain["P0"]
iv_atm = chain["iv"]

# Expiry from chain
expiry_ts = pd.Timestamp(chain["expiry"])
T_bt = (expiry_ts - entry_ts).days / 365.0
r_bt = 0.05

# --- 3) Estimate μ, σ from historical (e.g. 60-day lookback) ---
lookback = spy_df.loc[:entry_ts].tail(63)
log_ret = np.log(lookback["close"] / lookback["close"].shift(1)).dropna()
mu_annual = float(log_ret.mean() * 252)
sigma_annual = float(log_ret.std() * np.sqrt(252))
print(f"μ (annual)={mu_annual:.2%}  σ (annual)={sigma_annual:.2%}  IV={iv_atm:.2%}", flush=True)

# --- 4) Predicted μ, Σ from delta-gamma model ---
dt_period = holding_days / 252.0  # period in years
mu_pred, Sigma_pred = build_delta_gamma_moments(
    spot=spot_entry, mu=mu_annual, sigma=sigma_annual, r=r_bt, T=T_bt,
    K_call=K_call, K_put=K_put, C0=C0_mkt, P0=P0_mkt,
    dt=dt_period, sigma_greeks=iv_atm,  # use IV for Greeks
)
vol_pred = np.sqrt(np.maximum(np.diag(Sigma_pred), 0.0))

# --- 5) Realized ---
idx_today = spy_df.index.get_indexer([today], method="ffill")[0]
spot_today = float(spy_df.iloc[idx_today]["close"])
R_und_real = (spot_today - spot_entry) / spot_entry
if expiry_ts <= today:
    idx_exp = spy_df.index.get_indexer([expiry_ts], method="ffill")[0]
    spot_at_exp = float(spy_df.iloc[idx_exp]["close"])
else:
    spot_at_exp = spot_today
R_call_real = (max(spot_at_exp - K_call, 0.0) - C0_mkt) / C0_mkt
R_put_real = (max(K_put - spot_at_exp, 0.0) - P0_mkt) / P0_mkt
R_cash_real = np.exp(r_bt * holding_days / 365.0) - 1.0
mu_real = np.array([R_und_real, R_call_real, R_put_real, R_cash_real])

# --- 6) Realized cov from daily data (same as distribution.ipynb §6) ---
spy_slice = spy_df.loc[entry_ts:today]["close"]
spots = spy_slice.values.astype(float)
n_dates = len(spots)
if n_dates < 2:
    raise ValueError("Need at least 2 trading days")
dates_slice = spy_slice.index
T_remain_arr = np.array([max((expiry_ts - d).days / 365.0, 1e-6) for d in dates_slice])
call_vals = np.array([max(_bs_call_price(spots[i], K_call, r_bt, T_remain_arr[i], iv_atm), 0.01) for i in range(n_dates)])
put_vals = np.array([max(_bs_put_price(spots[i], K_put, r_bt, T_remain_arr[i], iv_atm), 0.01) for i in range(n_dates)])
r_und_d = (spots[1:] - spots[:-1]) / spots[:-1]
r_call_d = (call_vals[1:] - call_vals[:-1]) / call_vals[:-1]
r_put_d = (put_vals[1:] - put_vals[:-1]) / put_vals[:-1]
r_cash_d = (np.exp(r_bt / 365.0) - 1.0) * np.ones(n_dates - 1)
R_daily = np.column_stack([r_und_d, r_call_d, r_put_d, r_cash_d])
n_days = R_daily.shape[0]
Sigma_real = n_days * np.cov(R_daily, rowvar=False)
vol_real = np.std(R_daily, axis=0) * np.sqrt(n_days)
vol_real[3] = 0.0
Sigma_real[3, :] = 0.0
Sigma_real[:, 3] = 0.0
Sigma_real[3, 3] = 1e-12

# --- 7) Print comparison ---
print(f"\n{'='*70}", flush=True)
print(f"DELTA-GAMMA MODEL: {entry_date} → {today.date()} ({holding_days} days)", flush=True)
print(f"Entry spot: {spot_entry:.2f}  Today: {spot_today:.2f}", flush=True)
print(f"{'='*70}", flush=True)
print("\n--- Predicted vs realized expected returns ---", flush=True)
print(pd.DataFrame({"asset": ASSET_NAMES, "predicted": mu_pred, "realized": mu_real, "diff": mu_pred - mu_real}).to_string(index=False), flush=True)
print("\n--- Predicted vs realized vol ---", flush=True)
print(pd.DataFrame({"asset": ASSET_NAMES, "predicted_vol": vol_pred, "realized_vol": vol_real, "ratio": np.where(vol_real > 1e-12, vol_pred / vol_real, np.nan)}).to_string(index=False), flush=True)
print("\n--- Predicted covariance ---", flush=True)
print(pd.DataFrame(Sigma_pred, index=ASSET_NAMES, columns=ASSET_NAMES).round(6).to_string(), flush=True)
print("\n--- Realized covariance (daily, scaled) ---", flush=True)
print(pd.DataFrame(Sigma_real, index=ASSET_NAMES, columns=ASSET_NAMES).round(6).to_string(), flush=True)
print("\n--- Where realized falls in predicted ---", flush=True)
for i in range(3):
    # Approx percentile from normal (delta-gamma assumes approx normal returns)
    z = (mu_real[i] - mu_pred[i]) / (vol_pred[i] + 1e-12)
    pct = float(norm.cdf(z) * 100)
    print(f"  {ASSET_NAMES[i]}: realized {mu_real[i]:+.4f} ~ {pct:.1f}th percentile (z={z:.2f})", flush=True)

Entry date: 2026-02-25  (today=2026-02-26, holding=1 days)


ValueError: No expiries in chain

In [16]:
# --- 7) Rolling 1-day delta-gamma test (MODE A: fixed_symbol) ---

import numpy as np
import pandas as pd
from collections import defaultdict

CONTRACT_MODE = "fixed_symbol"        # only MODE A implemented
LAST_N_DAYS = 22                      # ~1 month of trading days
MAX_REL_SPREAD = 0.50                 # skip if spread/mid > 50%
THETA_IS_PER_DAY = False              # BS theta is per year; set True if using per-day theta
INCLUDE_VEGA_TERM = False             # optional vega * ΔIV extension
DEBUG = False                         # set True for verbose per-day diagnostics
DEBUG_Z_THRESHOLD = 3.0               # print debug when |z| > this

print(f"\nRolling 1-day test, CONTRACT_MODE={CONTRACT_MODE}")

# --- Helpers -------------------------------------------------------------

def _mid_from_row(row: pd.Series) -> float:
    bid = float(row.get("bid", np.nan))
    ask = float(row.get("ask", np.nan))
    last = float(row.get("lastprice", row.get("lastPrice", np.nan)))
    if bid > 0 and ask > 0:
        return 0.5 * (bid + ask)
    if last > 0:
        return last
    return np.nan

def _spread_from_row(row: pd.Series) -> float:
    bid = float(row.get("bid", np.nan))
    ask = float(row.get("ask", np.nan))
    if bid > 0 and ask > 0:
        return ask - bid
    return np.nan

def _load_chain_day(date_ts: pd.Timestamp):
    date_str = date_ts.strftime("%Y-%m-%d")
    calls = pd.read_parquet(OPTION_CHAINS_DIR / f"calls_{date_str}.parquet")
    puts = pd.read_parquet(OPTION_CHAINS_DIR / f"puts_{date_str}.parquet")
    # normalize columns
    calls = calls.rename(columns={"impliedVolatility": "impl_vol", "lastPrice": "lastprice"})
    puts = puts.rename(columns={"impliedVolatility": "impl_vol", "lastPrice": "lastprice"})
    return calls, puts

def _select_atm_rows(calls: pd.DataFrame, puts: pd.DataFrame, spot_t: float):
    call_atm = calls.loc[(calls["strike"] - spot_t).abs().idxmin()]
    put_atm = puts.loc[(puts["strike"] - spot_t).abs().idxmin()]
    return call_atm, put_atm

def _bs_vega(S: float, K: float, r: float, T: float, sigma: float) -> float:
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return 0.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return float(S * np.sqrt(T) * norm.pdf(d1))  # per 1.0 change in sigma

def _theta_per_day(theta_bs: float) -> float:
    # theta_bs is per year from _bs_theta
    if THETA_IS_PER_DAY:
        return theta_bs
    return theta_bs / 252.0

def _estimate_mu_sigma_daily(spy_df: pd.DataFrame, entry_ts: pd.Timestamp, lookback_days: int = 63):
    window = spy_df.loc[:entry_ts].tail(lookback_days)
    rets = window["close"].pct_change().dropna()
    if len(rets) < 10:
        return None, None, None
    mu_d = float(rets.mean())
    sig_d = float(rets.std(ddof=1))
    sig_ann = sig_d * np.sqrt(252)
    return mu_d, sig_d, sig_ann

def summarize(pred: pd.Series, real: pd.Series, name: str):
    err = pred - real
    mae = np.abs(err).mean()
    rmse = np.sqrt((err**2).mean())
    corr = np.corrcoef(pred, real)[0, 1] if len(pred) > 1 else np.nan
    print(
        f"{name}: mean_pred={pred.mean():+7.3%}  "
        f"mean_real={real.mean():+7.3%}  bias={err.mean():+7.3%}  "
        f"MAE={mae:7.3%}  RMSE={rmse:7.3%}  corr(pred,real)={corr:+.3f}"
    )

# --- Data prep -----------------------------------------------------------

spy_df = pd.read_parquet(SPY_DAILY_FILE)
spy_df.index = pd.to_datetime(spy_df.index).tz_localize(None).normalize()
spy_df = spy_df.sort_index()

# All chain dates we have
all_files = sorted(os.listdir(OPTION_CHAINS_DIR))
chain_dates = sorted({
    f.replace("calls_", "").replace("puts_", "").replace(".parquet", "")
    for f in all_files if f.startswith("calls_") or f.startswith("puts_")
})
chain_dates_ts = [pd.Timestamp(d) for d in chain_dates]
chain_dates_ts = sorted(chain_dates_ts)

chain_dates_recent = chain_dates_ts[-LAST_N_DAYS:]

rows = []
fail_counts = defaultdict(int)

# --- Main loop: 1-day steps ---------------------------------------------

for entry_ts in chain_dates_recent:
    date_str = entry_ts.strftime("%Y-%m-%d")
    if entry_ts not in spy_df.index:
        fail_counts["no_spy_bar"] += 1
        continue

    # next trading day
    pos = spy_df.index.get_loc(entry_ts)
    if isinstance(pos, slice):
        pos = pos.stop - 1
    if pos + 1 >= len(spy_df.index):
        fail_counts["no_next_day"] += 1
        continue
    next_ts = spy_df.index[pos + 1]
    date_str_1 = next_ts.strftime("%Y-%m-%d")

    spot_t = float(spy_df.loc[entry_ts, "close"])
    spot_t1 = float(spy_df.loc[next_ts, "close"])

    # load chains
    try:
        calls_t, puts_t = _load_chain_day(entry_ts)
        calls_t1, puts_t1 = _load_chain_day(next_ts)
    except FileNotFoundError:
        fail_counts["missing_chain_files"] += 1
        continue

    if len(calls_t) == 0 or len(puts_t) == 0:
        fail_counts["empty_chain_t"] += 1
        continue

    # MODE A: fixed_symbol
    try:
        call_atm_t, put_atm_t = _select_atm_rows(calls_t, puts_t, spot_t)
    except Exception:
        fail_counts["atm_select_error"] += 1
        continue

    call_sym = call_atm_t.get("contractSymbol", None)
    put_sym = put_atm_t.get("contractSymbol", None)
    if call_sym is None or put_sym is None:
        fail_counts["missing_contract_symbol"] += 1
        continue

    # same symbols at t+1 (fallback: match by expiry+strike if symbol missing)
    call_t1_rows = calls_t1[calls_t1["contractSymbol"] == call_sym]
    put_t1_rows = puts_t1[puts_t1["contractSymbol"] == put_sym]
    if call_t1_rows.empty:
        call_t1_rows = calls_t1[(calls_t1["expiry"] == call_atm_t["expiry"]) & (np.isclose(calls_t1["strike"], float(call_atm_t["strike"])))]
    if put_t1_rows.empty:
        put_t1_rows = puts_t1[(puts_t1["expiry"] == put_atm_t["expiry"]) & (np.isclose(puts_t1["strike"], float(put_atm_t["strike"])))]
    if call_t1_rows.empty or put_t1_rows.empty:
        fail_counts["missing_same_symbol_next_day"] += 1
        continue

    call_atm_t1 = call_t1_rows.iloc[0]
    put_atm_t1 = put_t1_rows.iloc[0]

    # mids and spreads at t, t+1
    C_t = _mid_from_row(call_atm_t)
    P_t = _mid_from_row(put_atm_t)
    C_t1 = _mid_from_row(call_atm_t1)
    P_t1 = _mid_from_row(put_atm_t1)

    if not np.isfinite(C_t) or C_t <= 0 or not np.isfinite(P_t) or P_t <= 0:
        fail_counts["bad_mid_t"] += 1
        continue
    if not np.isfinite(C_t1) or C_t1 <= 0 or not np.isfinite(P_t1) or P_t1 <= 0:
        fail_counts["bad_mid_t1"] += 1
        continue

    spread_C_t = _spread_from_row(call_atm_t)
    spread_P_t = _spread_from_row(put_atm_t)
    rel_spread_C = spread_C_t / C_t if spread_C_t is not None and C_t > 0 else np.nan
    rel_spread_P = spread_P_t / P_t if spread_P_t is not None and P_t > 0 else np.nan

    if np.isfinite(rel_spread_C) and rel_spread_C > MAX_REL_SPREAD:
        fail_counts["spread_too_wide_call"] += 1
        continue
    if np.isfinite(rel_spread_P) and rel_spread_P > MAX_REL_SPREAD:
        fail_counts["spread_too_wide_put"] += 1
        continue

    # expiry / DTE
    expiry_c = pd.Timestamp(call_atm_t["expiry"])
    expiry_p = pd.Timestamp(put_atm_t["expiry"])
    # require both expiries same for clean test
    if expiry_c != expiry_p:
        fail_counts["mismatched_expiry_call_put"] += 1
        continue
    expiry_ts = expiry_c
    T_entry = max((expiry_ts - entry_ts).days / 365.0, 1e-6)
    T_next = max((expiry_ts - next_ts).days / 365.0, 0.0)
    dte_c = (expiry_ts - entry_ts).days

    # daily μ, σ
    mu_d, sig_d, sig_ann = _estimate_mu_sigma_daily(spy_df, entry_ts)
    if mu_d is None:
        fail_counts["too_short_lookback"] += 1
        continue

    # BS IV at t (from chain 'impl_vol' if present; else invert)
    Kc = float(call_atm_t["strike"])
    Kp = float(put_atm_t["strike"])
    iv_c = float(call_atm_t.get("impl_vol", np.nan))
    iv_p = float(put_atm_t.get("impl_vol", np.nan))
    r = 0.05

    if not np.isfinite(iv_c) or iv_c <= 0:
        iv_c = _bs_implied_vol(C_t, spot_t, Kc, r, T_entry, is_call=True) or 0.20
    if not np.isfinite(iv_p) or iv_p <= 0:
        iv_p = _bs_implied_vol(P_t, spot_t, Kp, r, T_entry, is_call=False) or 0.20
    iv_t = float(np.mean([iv_c, iv_p]))

    # Greeks at t from BS (per-year theta from helper)
    dc = _bs_delta(spot_t, Kc, r, T_entry, iv_t, is_call=True)
    dp = _bs_delta(spot_t, Kp, r, T_entry, iv_t, is_call=False)
    gc = _bs_gamma(spot_t, Kc, r, T_entry, iv_t)
    gp = _bs_gamma(spot_t, Kp, r, T_entry, iv_t)
    tc_year = _bs_theta(spot_t, Kc, r, T_entry, iv_t, is_call=True)
    tp_year = _bs_theta(spot_t, Kp, r, T_entry, iv_t, is_call=False)
    tc = _theta_per_day(tc_year)
    tp = _theta_per_day(tp_year)
    vc = _bs_vega(spot_t, Kc, r, T_entry, iv_t)
    vp = _bs_vega(spot_t, Kp, r, T_entry, iv_t)

    # ΔIV for optional vega term
    iv_c1 = float(call_atm_t1.get("impl_vol", np.nan))
    iv_p1 = float(put_atm_t1.get("impl_vol", np.nan))
    if not np.isfinite(iv_c1) or iv_c1 <= 0:
        iv_c1 = _bs_implied_vol(C_t1, spot_t1, Kc, r, T_next, is_call=True) or iv_t
    if not np.isfinite(iv_p1) or iv_p1 <= 0:
        iv_p1 = _bs_implied_vol(P_t1, spot_t1, Kp, r, T_next, is_call=False) or iv_t
    iv_t1 = float(np.mean([iv_c1, iv_p1]))
    dIV = iv_t1 - iv_t

    # --- Predicted 1-day ΔC, ΔP (delta + theta + gamma [+ vega]) ----
    dt = 1.0 / 252.0

    dC_delta = dc * mu_d * spot_t
    dC_theta = tc
    dC_gamma = 0.5 * gc * (sig_d ** 2) * (spot_t ** 2)
    dC_vega = vc * dIV if INCLUDE_VEGA_TERM else 0.0
    dC_pred = (dC_delta + dC_theta + dC_gamma + dC_vega)  # per day

    dP_delta = dp * mu_d * spot_t
    dP_theta = tp
    dP_gamma = 0.5 * gp * (sig_d ** 2) * (spot_t ** 2)
    dP_vega = vp * dIV if INCLUDE_VEGA_TERM else 0.0
    dP_pred = (dP_delta + dP_theta + dP_gamma + dP_vega)

    mu_call_pred_1d = dC_pred / C_t
    mu_put_pred_1d = dP_pred / P_t
    mu_und_pred_1d = mu_d
    mu_cash_pred_1d = np.exp(r * dt) - 1.0

    # Predicted std (approx) from delta leverage
    std_und_pred_1d = sig_d
    std_call_pred_1d = abs(dc) * sig_d * (spot_t / C_t)
    std_put_pred_1d = abs(dp) * sig_d * (spot_t / P_t)

    # --- Realized 1-day returns (MODE A: same symbols) ---------------
    R_und_real_1d = (spot_t1 - spot_t) / spot_t
    R_call_real_1d = (C_t1 - C_t) / C_t
    R_put_real_1d = (P_t1 - P_t) / P_t
    R_cash_real_1d = np.exp(r * dt) - 1.0

    z_call = (R_call_real_1d - mu_call_pred_1d) / (std_call_pred_1d + 1e-12)
    z_put = (R_put_real_1d - mu_put_pred_1d) / (std_put_pred_1d + 1e-12)

    if DEBUG or (abs(z_call) > DEBUG_Z_THRESHOLD or abs(z_put) > DEBUG_Z_THRESHOLD):
        print(f"\n[{date_str}] → [{date_str_1}] spot={spot_t:.2f}→{spot_t1:.2f}  dte={dte_c}")
        print(f"CALL {call_sym} K={Kc:.1f} exp={expiry_ts.date()}  C={C_t:.4f}→{C_t1:.4f}  rel_spread={rel_spread_C:.2%}")
        print(f"PUT  {put_sym}  K={Kp:.1f} exp={expiry_ts.date()}  P={P_t:.4f}→{P_t1:.4f}  rel_spread={rel_spread_P:.2%}")
        print(f"mu_d={mu_d:+.6f}  sig_d={sig_d:.6f}  sig_ann≈{sig_ann:.2%}")
        print(f"CALL greeks: delta={dc:+.4f} gamma={gc:.6g} theta_day={tc:+.6g} (theta_per_day={THETA_IS_PER_DAY}) vega={vc:.6g}")
        print(f"PUT  greeks: delta={dp:+.4f} gamma={gp:.6g} theta_day={tp:+.6g} (theta_per_day={THETA_IS_PER_DAY}) vega={vp:.6g}")
        print(f"CALL dC parts: delta*mu*S={dC_delta:+.6g} theta={dC_theta:+.6g} 0.5*gamma*s2*S2={dC_gamma:+.6g} vega*dIV={dC_vega:+.6g} total={dC_pred:+.6g}  mu_pred={mu_call_pred_1d:+.3%}")
        print(f"PUT  dP parts: delta*mu*S={dP_delta:+.6g} theta={dP_theta:+.6g} 0.5*gamma*s2*S2={dP_gamma:+.6g} vega*dIV={dP_vega:+.6g} total={dP_pred:+.6g}  mu_pred={mu_put_pred_1d:+.3%}")
        print(f"Realized: R_und={R_und_real_1d:+.3%}  R_call={R_call_real_1d:+.3%}  R_put={R_put_real_1d:+.3%}")
        print(f"PredStd : und={std_und_pred_1d:.3%}  call≈{std_call_pred_1d:.3f}  put≈{std_put_pred_1d:.3f}  z_call={z_call:+.2f}  z_put={z_put:+.2f}")
        if INCLUDE_VEGA_TERM:
            print(f"IV_t={iv_t:.4f} → IV_t1={iv_t1:.4f}  dIV={dIV:+.4f}")

    rows.append({
        "date": date_str,
        "mu_und_pred_1d": mu_und_pred_1d,
        "mu_call_pred_1d": mu_call_pred_1d,
        "mu_put_pred_1d": mu_put_pred_1d,
        "mu_cash_pred_1d": mu_cash_pred_1d,
        "R_und_real_1d": R_und_real_1d,
        "R_call_real_1d": R_call_real_1d,
        "R_put_real_1d": R_put_real_1d,
        "R_cash_real_1d": R_cash_real_1d,
        "std_und_pred_1d": std_und_pred_1d,
        "std_call_pred_1d": std_call_pred_1d,
        "std_put_pred_1d": std_put_pred_1d,
    })

df_roll = pd.DataFrame(rows)

print(f"\nRolling 1-day test over last {LAST_N_DAYS} chain dates (successful={len(df_roll)}, failures={dict(fail_counts)})")

if not df_roll.empty:
    print("\nSample of per-day predicted vs realized:")
    print(df_roll[[
        "date",
        "mu_und_pred_1d", "R_und_real_1d",
        "mu_call_pred_1d", "R_call_real_1d",
        "mu_put_pred_1d", "R_put_real_1d",
    ]].head(10).to_string(index=False))

    print("\nMean/bias/MAE/RMSE + corr(pred,real):")
    summarize(df_roll["mu_und_pred_1d"],  df_roll["R_und_real_1d"],  "UNDERLYING")
    summarize(df_roll["mu_call_pred_1d"], df_roll["R_call_real_1d"], "CALL")
    summarize(df_roll["mu_put_pred_1d"],  df_roll["R_put_real_1d"],  "PUT")
    summarize(df_roll["mu_cash_pred_1d"], df_roll["R_cash_real_1d"], "CASH")

    print("\nVariance calibration (predicted std vs realized std of returns):")
    for name, pred_std_col, real_col in [
        ("UNDERLYING", "std_und_pred_1d", "R_und_real_1d"),
        ("CALL",       "std_call_pred_1d", "R_call_real_1d"),
        ("PUT",        "std_put_pred_1d",  "R_put_real_1d"),
    ]:
        pred_std = df_roll[pred_std_col]
        real_abs = df_roll[real_col].abs()
        corr_std_abs = np.corrcoef(pred_std, real_abs)[0, 1] if len(pred_std) > 1 else np.nan
        print(
            f"{name}: mean_pred_std={pred_std.mean():.4f}  "
            f"realized_std={df_roll[real_col].std(ddof=1):.4f}  "
            f"corr(pred_std, |real|)={corr_std_abs:+.3f}"
        )
else:
    print("No valid days for rolling test; see")


Rolling 1-day test, CONTRACT_MODE=fixed_symbol

[2026-02-05] → [2026-02-06] spot=677.62→690.62  dte=8
CALL SPY260213C00678000 K=678.0 exp=2026-02-13  C=8.6100→16.0600  rel_spread=nan%
PUT  SPY260213P00678000  K=678.0 exp=2026-02-13  P=8.4200→2.1700  rel_spread=nan%
mu_d=+0.000027  sig_d=0.007231  sig_ann≈11.48%
CALL greeks: delta=+0.5131 gamma=0.018678 theta_day=-0.837796 (theta_per_day=False) vega=40.0002
PUT  greeks: delta=-0.4869 gamma=0.018678 theta_day=-0.703419 (theta_per_day=False) vega=40.0002
CALL dC parts: delta*mu*S=+0.00929541 theta=-0.837796 0.5*gamma*s2*S2=+0.224199 vega*dIV=+0 total=-0.604301  mu_pred=-7.019%
PUT  dP parts: delta*mu*S=-0.00882218 theta=-0.703419 0.5*gamma*s2*S2=+0.224199 vega*dIV=+0 total=-0.488042  mu_pred=-5.796%
Realized: R_und=+1.918%  R_call=+86.527%  R_put=-74.228%
PredStd : und=0.723%  call≈0.292  put≈0.283  z_call=+3.20  z_put=-2.42

Rolling 1-day test over last 22 chain dates (successful=7, failures={'missing_same_symbol_next_day': 13, 'empty_c

/Users/hakeemshindy/miniconda3/envs/cvx_options/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
